## Project Description

For the Words to Embeddings group work assignment, we decided to create word embeddings on a small-, medium-, and large-sized dataset that builds off the same corpus of text and then test to see how relationships between words differ between the sets of vectors. The corpus of text we used is The Complete Works of Shakespeare pulled from Project Gutenberg. The small dataset consists of the first 25% of the plain text file, the medium dataset consists of the first 50% of the file, and the large dataset is the entire file.

After all of the embeddings were created and trained with each sized dataset, we tested to see how certain relationships between words differed. The testing consisted of using the vector offset method to get the word embedding for "queen" from the expression "king - man + woman" and seeing how off each set of embeddings was from the true vector for "queen". 

## Results

For each test, each model is loaded separately and the vector for the word we are aiming for is pulled from the model. We then use vector offset to calcucate the wanted vector and compare it with the true vector with cosine similarity.

Model Evaluation on: 'king - man + woman = queen'
- Small text model: 0.5798753499984741
- Medium text model: 0.6685549020767212
- Large text model: 0.7705735564231873

Model Evaluation on: 'good - better + bad = worse'
- Small text model: 0.304522305727005
- Medium text model: 0.24335289001464844
- Large text model: 0.32541072368621826

Model Evaluation on: 'hamlet - man + woman = ophelia'
- Small text model: 0.42612141370773315
- Medium text model: 0.5201843976974487
- Large text model: 0.6225850582122803

Model Evaluation on: 'father - man + woman = mother'
- Small text model: 0.655929446220398
- Medium text model: 0.5369539856910706
- Large text model: 0.6917486786842346

Model Evaluation on: 'prince - man + woman = princess'
- Small text model: 0.24032637476921082
- Medium text model: 0.26789969205856323
- Large text model: 0.5025824308395386

Model Evaluation on: 'son - man + woman = daughter'
- Small text model: 0.3943868577480316
- Medium text model: 0.5509151816368103
- Large text model: 0.7645407319068909

Model Evaluation on: 'father - son + daughter = mother'
- Small text model: 0.4222005605697632
- Medium text model: 0.4654947519302368
- Large text model: 0.6168953776359558

Model Evaluation on: 'love - man + woman = love'
- Small text model: 0.6932451725006104
- Medium text model: 0.6965163946151733
- Large text model: 0.8168501853942871

Based off of the testing results, for the most part the larger the text file used for training the better the embeddings were able to capture relationships between words. There was a couple trials that provided interesting results that did not follow this trend; however, the relationships being tested in those trials were a bit more arbitrary compared to the gender and royalty relationships that were tested in the other trials. Overall, we would have liked to seen better similarity results especially from the large-sized text model as most trails resulted in around 0.6 similarity to the true vector. Also, the word we were aiming for many of the trials was not the most similar word to the calculated vector.

#### Dependencies

Installing all dependencies needed for the rest of the file.

In [93]:
from gensim.utils import simple_preprocess
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DSU\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#### Helper Functions

These two functions help create the sentence tokens the skip-gram model needs to train correctly and create the skip-gram model and train it.

Both of these functions are made up of code generated by ChatGPT.

In [ ]:
def create_sentence_tokens(text):
    sentences = sent_tokenize(text) 
    sentence_tokens = [simple_preprocess(sent) for sent in sentences]
    return sentence_tokens

def create_model(title, sentences):
    model = Word2Vec(
        sentences=sentences,
        vector_size=50,   
        window=5,
        min_count=2,       
        sg=1,              # 1 = skip-gram, 0 = CBOW
        negative=10,
        sample=1e-4,
        epochs=500
    )

    model.save(f'{title}.model')

#### Example Testing

A few lines used to separatly test the Word2Vec models. Was used to see how well the first few models did with training their embeddings.

In [97]:
model = Word2Vec.load("lg.model")

import numpy as np

eval_vec = model.wv['king'] - model.wv['man'] + model.wv['woman']

# find closest word
similar = model.wv.similar_by_vector(eval_vec, topn=5)
print(similar)


[('king', 0.8242897391319275), ('daughter', 0.7740620970726013), ('queen', 0.7705735564231873), ('son', 0.7692555785179138), ('henry', 0.7475244998931885)]


#### Large-Sized Model Creation

Used the helper functions above to create a skip-gram model with the large-sized text subset(the entire text file). It then saves the model as 'lg.model'.

In [ ]:
text = Path("Shakespeare.txt").read_text(encoding='utf-8', errors='ignore')

sentence_tokens = create_sentence_tokens(text)

create_model("lg", sentence_tokens)

#### Small-Sized Model Creation

Used the helper functions above to create a skip-gram model with the small-sized text subset(the first 25% of the text file). It then saves the model as 'sm.model'.

In [ ]:
file_path = "Shakespeare.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# take first 25%
cutoff = int(len(lines) * 0.25)
sm_text = "".join(lines[:cutoff])

sentence_tokens = create_sentence_tokens(sm_text)

create_model("sm", sentence_tokens)

#### Medium-Sized Model Creation

Used the helper functions above to create a skip-gram model with the medium-sized text subset(the first 50% of the text file). It then saves the model as 'md.model'.

In [ ]:
file_path = "Shakespeare.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# take first 50%
cutoff = int(len(lines) * 0.50)
md_text = "".join(lines[:cutoff])

sentence_tokens = create_sentence_tokens(md_text)

create_model("md", sentence_tokens)

#### Model Evaluation Helper Function

This function helps with evaluating and comparing the word embeddings created from each sized model. The function takes in 4 words and evaluates the expession 'a - b + c = d' for each model. For the evaluation, the vector offset operation 'a - b + c' is saved as well as the vector for the correct word 'd'. The two vectors are then compared using cosine similarity and the results are printed for each model.

In [ ]:
def evaluate_models(a, b, c, d):
    print(f"Model Evaluation on: '{a} - {b} + {c} = {d}'")

    model = Word2Vec.load("sm.model")
    eval_vec = model.wv[a] - model.wv[b] + model.wv[c]
    d_vec = model.wv[d]
    similarity = cosine_similarity([eval_vec], [d_vec])[0][0]
    print(f"Small text model: {similarity}")

    model = Word2Vec.load("md.model")
    eval_vec = model.wv[a] - model.wv[b] + model.wv[c]
    d_vec = model.wv[d]
    similarity = cosine_similarity([eval_vec], [d_vec])[0][0]
    print(f"Medium text model: {similarity}")

    model = Word2Vec.load("lg.model")
    eval_vec = model.wv[a] - model.wv[b] + model.wv[c]
    d_vec = model.wv[d]
    similarity = cosine_similarity([eval_vec], [d_vec])[0][0]
    print(f"Large text model: {similarity}")

#### Model Evaluation Trials

Below are all the evaluation trials that were used to compare the different sized models. The helper function above was used to evaluate and print the results from each trial.

In [ ]:
evaluate_models('king', 'man', 'woman', 'queen')
evaluate_models('good', 'better', 'bad', 'worse')
evaluate_models('hamlet', 'man', 'woman', 'ophelia')
evaluate_models('father', 'man', 'woman', 'mother')
evaluate_models('prince', 'man', 'woman', 'princess')
evaluate_models('son', 'man', 'woman', 'daughter')
evaluate_models('father', 'son', 'daughter', 'mother')
evaluate_models('love', 'man', 'woman', 'love')

Model Evaluation on: 'king - man + woman = queen'
Small text model: 0.5798753499984741
Medium text model: 0.6685549020767212
Large text model: 0.7705735564231873
Model Evaluation on: 'good - better + bad = worse'
Small text model: 0.304522305727005
Medium text model: 0.24335289001464844
Large text model: 0.32541072368621826
Model Evaluation on: 'hamlet - man + woman = ophelia'
Small text model: 0.42612141370773315
Medium text model: 0.5201843976974487
Large text model: 0.6225850582122803
Model Evaluation on: 'father - man + woman = mother'
Small text model: 0.655929446220398
Medium text model: 0.5369539856910706
Large text model: 0.6917486786842346
Model Evaluation on: 'prince - man + woman = princess'
Small text model: 0.24032637476921082
Medium text model: 0.26789969205856323
Large text model: 0.5025824308395386
Model Evaluation on: 'son - man + woman = daughter'
Small text model: 0.3943868577480316
Medium text model: 0.5509151816368103
Large text model: 0.7645407319068909
Model Eval